# 02 — match win/draw/loss (experiment)

The live site already has Play + 300 rolls. This notebook is extra ML. It is **not** a tip.

Use **finished** matches only. Features you actually have: home/away, goals scored/conceded coming in, form (W/D/L). Skip shots/possession until you have a licensed source.

Start with `RandomForestClassifier`. Skip XGBoost until this works on Windows.

Train on older gameweeks. Test on later **finished** gameweeks. Never test on games that have not been played if you want an honest number.

Write one sentence: “On the last N finished matches, this toy model was right X% of the time.”

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

load_dotenv(Path(".env"))
TOKEN = os.getenv("FOOTBALL_DATA_TOKEN", "").strip()
print("Token loaded:", bool(TOKEN))
print("You can also use finished FPL fixtures the website already maps. Either source is fine if it is public/licensed.")

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Next session: fill this with finished matches only.
# Columns you actually have, e.g. home_id, gf_in, ga_in, form points — not shots unless licensed.
matches = pd.DataFrame(
    columns=["gameweek", "home", "away", "result"]  # result in {H, D, A}
)
matches

In [ ]:
if matches.empty or matches["result"].isna().all():
    print("No finished matches loaded yet. Fill the table, then train.")
else:
    cut = int(matches["gameweek"].max() * 0.7)
    train = matches[matches["gameweek"] <= cut]
    test = matches[matches["gameweek"] > cut]
    # Replace feature_cols with columns you actually built.
    feature_cols = [c for c in train.columns if c not in {"result", "gameweek", "home", "away"}]
    if not feature_cols:
        print("Add numeric features (form, gf/ga coming in) before fitting.")
    else:
        model = RandomForestClassifier(n_estimators=200, random_state=17)
        model.fit(train[feature_cols], train["result"])
        pred = model.predict(test[feature_cols])
        acc = accuracy_score(test["result"], pred)
        n = len(test)
        print(
            f"On the last {n} finished matches, this toy model was right {acc:.0%} of the time. Not a tip."
        )

Later on Play: a small box “History check: X% on past weeks. Not a tip.”

```text
pip install xgboost
```

If that fails on Windows, stay on Random Forest.